# Explore a flight in Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AFNASA-ULI/flightdata/blob/main/notebooks/explore_flight.ipynb)

Loads the **processed** data for one flight: `processed/<flight>/flight.parquet`, `wind.parquet` and
`metadata.json`. The flight columns are the standard set listed on the *Processed data* page, so this works
the same for every drone and log format.

Run it from a clone of the repository (`pip install pandas pyarrow matplotlib`), or open it in Colab, where it
reads the files straight from GitHub.

In [ ]:
import io
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_RAW = "https://github.com/AFNASA-ULI/flightdata/raw/main"
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "processed").is_dir()), None)


def fetch(rel):
    """Bytes of a repository file: from the local clone if there is one, else from GitHub."""
    if ROOT is not None:
        p = ROOT / rel
        return p.read_bytes() if p.exists() else None
    try:
        with urllib.request.urlopen(f"{REPO_RAW}/{rel}") as r:
            return r.read()
    except OSError:
        return None


print("reading from", ROOT or REPO_RAW)

Pick a flight. The flight IDs are the folder names under `processed/` (and on the website's flight pages).

In [ ]:
FLIGHT = "Tarot450_20260925_165053Z"

meta = json.loads(fetch(f"processed/{FLIGHT}/metadata.json"))
df = pd.read_parquet(io.BytesIO(fetch(f"processed/{FLIGHT}/flight.parquet")))
wind_bytes = fetch(f"processed/{FLIGHT}/wind.parquet")
wind = pd.read_parquet(io.BytesIO(wind_bytes)) if wind_bytes else None
print(meta["platform"]["label"], meta["summary"]["start_utc"], df.shape,
      "wind rows:", 0 if wind is None else len(wind))
meta["summary"]

Data-quality findings from the pipeline:

In [ ]:
for c in meta["checks"]:
    print(f"[{c['level']}] {c['title']}: {c['detail']}")

Column units and descriptions:

In [ ]:
pd.DataFrame(meta["columns"]).T

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(df.elapsed_s, df.alt_rel_m); ax[0].set_ylabel("alt rel [m]")
ax[1].plot(df.elapsed_s, df.groundspeed_m_s, label="groundspeed")
ax[1].plot(df.elapsed_s, df.climb_m_s, label="climb"); ax[1].set_ylabel("m/s"); ax[1].legend()
ax[2].plot(df.elapsed_s, df.roll_deg, label="roll")
ax[2].plot(df.elapsed_s, df.pitch_deg, label="pitch"); ax[2].set_ylabel("deg"); ax[2].legend()
ax[-1].set_xlabel("elapsed [s]")
for a in ax:
    for t0, t1, mode in meta["summary"]["modes"]:
        if mode == "AUTO":
            a.axvspan(t0, t1, color="C0", alpha=0.06)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df.east_m, df.north_m, c=df.elapsed_s, s=2)
plt.axis("equal"); plt.xlabel("east [m]"); plt.ylabel("north [m]"); plt.colorbar(label="elapsed [s]")

Wind sensor, already trimmed to this flight and on the same `elapsed_s` axis:

In [ ]:
if wind is not None and len(wind):
    fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    ax[0].plot(wind.elapsed_s, wind.wind_speed_m_s); ax[0].set_ylabel("wind [m/s]")
    ax[1].plot(wind.elapsed_s, wind.wind_dir_deg, ".", ms=2); ax[1].set_ylabel("direction [deg]")
    ax[1].set_xlabel("elapsed [s]"); plt.tight_layout()
else:
    print("No wind data overlaps this flight.")

Wind at each flight sample (nearest wind reading within 0.5 s):

In [ ]:
if wind is not None and len(wind):
    joined = pd.merge_asof(df.sort_values("time_gps_utc"), wind[["time_utc", "wind_speed_m_s", "wind_dir_deg"]],
                           left_on="time_gps_utc", right_on="time_utc", direction="nearest",
                           tolerance=pd.Timedelta("0.5s"), suffixes=("", "_wind"))
    display(joined[["elapsed_s", "groundspeed_m_s", "pitch_deg", "wind_speed_m_s", "wind_dir_deg"]].describe())